In [1]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import warnings

# --- ML & Data ---
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, concatenate
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical

# --- Transformers (BERT) ---
from transformers import DistilBertTokenizer, TFDistilBertForSequenceClassification

# --- Setup ---
warnings.filterwarnings('ignore')
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
stop_words = set(stopwords.words('english'))

print("All libraries imported.")
print(f"TensorFlow Version: {tf.__version__}")

2026-02-13 05:10:50.334563: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770959450.756331      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770959450.874431      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770959451.806100      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770959451.806149      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770959451.806152      55 computation_placer.cc:177] computation placer alr

All libraries imported.
TensorFlow Version: 2.19.0


In [2]:
file_name="/kaggle/input/fakejobss/fake_job_postings.csv"
df=pd.read_csv(file_name)

In [3]:
text_columns = ['title', 'company_profile', 'description', 'requirements', 'benefits']
for col in text_columns:
    df[col] = df[col].fillna('')

df['text_combined'] = df[text_columns].apply(lambda x: ' '.join(x), axis=1)

In [4]:
df.head()

,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent,text_combined
0,1,Marketing Intern,"US, NY, New York",Marketing,NaN,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,,0,1,0,Other,Internship,NaN,NaN,Marketing,0,"Marketing Intern We're Food52, and we've creat..."
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,1,0,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,0,Customer Service - Cloud Video Production 90 S...
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,,0,1,0,NaN,NaN,NaN,NaN,NaN,0,Commissioning Machinery Assistant (CMA) Valor ...
3,4,Account Executive - Washington DC,"US, DC, Washington",Sales,NaN,Our passion for improving quality of life thro...,THE COMPANY: ESRI – Environmental Systems Rese...,"EDUCATION: Bachelor’s or Master’s in GIS, busi...",Our culture is anything but corporate—we have ...,0,1,0,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,0,Account Executive - Washington DC Our passion ...
4,5,Bill Review Manager,"US, FL, Fort Worth",NaN,NaN,SpotSource Solutions LLC is a Global Human Cap...,JOB TITLE: Itemization Review ManagerLOCATION:...,QUALIFICATIONS:RN license in the State of Texa...,Full Benefits Offered,0,1,1,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,0,Bill Review Manager SpotSource Solutions LLC i...


In [5]:
metadata_features = ['employment_type', 'required_experience', 'required_education', 'industry', 'function']
df[metadata_features] = df[metadata_features].fillna('Missing')

encoders = {}
for col in metadata_features:
    le = LabelEncoder()
    df[col + '_encoded'] = le.fit_transform(df[col])
    encoders[col] = le

In [6]:
metadata_vocab_sizes = {col: len(encoders[col].classes_) for col in metadata_features}
print(f"Metadata encoded. Vocab sizes: {metadata_vocab_sizes}")


target = 'fraudulent'
y = df[target].values

print("✅ Data preprocessing complete.")
print(df[['text_combined', 'employment_type_encoded', 'fraudulent']].head())

Metadata encoded. Vocab sizes: {'employment_type': 6, 'required_experience': 8, 'required_education': 14, 'industry': 132, 'function': 38}
✅ Data preprocessing complete.
                                       text_combined  employment_type_encoded  \
0  Marketing Intern We're Food52, and we've creat...                        3   
1  Customer Service - Cloud Video Production 90 S...                        1   
2  Commissioning Machinery Assistant (CMA) Valor ...                        2   
3  Account Executive - Washington DC Our passion ...                        1   
4  Bill Review Manager SpotSource Solutions LLC i...                        1   

   fraudulent  
0           0  
1           0  
2           0  
3           0  
4           0  


In [7]:
# Define the categories and keywords to search for
categories = {
    "Data Analyst": "Data Analyst",
    "Software Developer": "Software",  # 'Software' captures Developer/Engineer titles
    "Finance": "Finance",
    "Marketing Manager": "Marketing Manager"
}

# List to store the selected rows
selected_jobs = []

for category, keyword in categories.items():
    # Filter rows where the title contains the keyword (case-insensitive)
    # and take the first match found
    job = df[df['title'].str.contains(keyword, case=False, na=False)].iloc[0]
    selected_jobs.append(job)

# Create a DataFrame from the selected rows for clean display
examples_df = pd.DataFrame(selected_jobs)

# Display the specific columns you are interested in
columns_to_show = ['title', 'text_combined', 'employment_type_encoded', 'fraudulent']
print(examples_df[columns_to_show])

                                               title  \
753                                     Data Analyst   
31                  Software Applications Specialist   
64   SENIOR FINANCE SOFTWARE RESEARCHER AND ENGINEER   
55                       Junior HR Marketing Manager   

                                         text_combined  \
753  Data Analyst Growing out of Forward Internet G...   
31   Software Applications Specialist  Day to Day-I...   
64   SENIOR FINANCE SOFTWARE RESEARCHER AND ENGINEE...   
55   Junior HR Marketing Manager We are Netguru and...   

     employment_type_encoded  fraudulent  
753                        1           0  
31                         1           0  
64                         2           0  
55                         2           0  


In [8]:
# Assuming your filtered dataframe is named 'examples_df'
# If you are selecting directly from the main df, use: selected_jobs = df.loc[[753, 31, 64, 55]]

for index, row in examples_df.iterrows():
    print(f"--- Job Title: {row['title']} ---")
    print(row['text_combined'])  # You can change this to row['description'] if you prefer
    print("\n" + "="*80 + "\n")  # Prints a separator line

--- Job Title: Data Analyst ---
Data Analyst Growing out of Forward Internet Group, Scramble has 6 years’ experience of running Internet marketing campaigns. Our focus is on developing smart, automated solutions to maintain our track record of aggressive growth in an ever-more competitive advertising space. We are looking for a Data Analyst to join our growing team to develop and manage PPC accounts.Your day-to-day responsibilities will include using data management tools like Excel and MySQL to review and analyse Adwords data in order to improve the performance of our campaigns.  You will collaborate closely with the rest of the team to help to create new processes and identify opportunities to grow our business. Essential skills and Experience●      Excellent Excel skills; including pivot tables, graphing, vlookups, logic statements and formatting.●      Adwords account management; knowledge of PPC best practice and account performance●      Data analytics; able to query, extract and

In [ ]:
# 1. Create a mapping dictionary
status_map = {0: "Real", 1: "Fake"}

# 2. Apply it to your specific examples (assuming you have 'examples_df' from the previous step)
examples_df['status'] = examples_df['fraudulent'].map(status_map)

# 3. Display the Title and Status
print(examples_df[['title', 'status']])

In [9]:
import pandas as pd

# 1. Set display options to see full text
pd.set_option('display.max_colwidth', None)

# 2. Define categories and keywords
categories = {
    "Data Analyst": "Data Analyst",
    "Software Developer": "Software",
    "Finance": "Finance",
    "Marketing": "Marketing"
}

all_selected_jobs = []

for cat_name, keyword in categories.items():
    # Filter the main dataframe for the keyword in the title
    category_subset = df[df['title'].str.contains(keyword, case=False, na=False)]
    
    # Try to get 2 Real jobs and 2 Fake jobs to ensure a mix
    real_jobs = category_subset[category_subset['fraudulent'] == 0].head(2)
    fake_jobs = category_subset[category_subset['fraudulent'] == 1].head(2)
    
    # Combine them for this category
    combined = pd.concat([real_jobs, fake_jobs])
    all_selected_jobs.append(combined)

# 3. Create the final display dataframe
mixed_results_df = pd.concat(all_selected_jobs).drop_duplicates()

# 4. Add a readable status column
mixed_results_df['status'] = mixed_results_df['fraudulent'].map({0: '✅ Real', 1: '🚩 Fake'})

# 5. Display the results
# Note: Since some categories might not have many 'Fake' jobs, 
# it will show as many as it can find (usually 3-4 per category).
print(mixed_results_df[['title', 'status', 'text_combined']])

                                                            title  status  \
753                                                  Data Analyst  ✅ Real   
894                              Scandinavian Equity Data Analyst  ✅ Real   
31                               Software Applications Specialist  ✅ Real   
64                SENIOR FINANCE SOFTWARE RESEARCHER AND ENGINEER  ✅ Real   
1854                                 Automation Software Engineer  🚩 Fake   
2367               Software Engineer | Forecasting & Optimization  🚩 Fake   
323   Partnership Manager - High Growth Specialty Finance Company  ✅ Real   
4939                                   Position Finance Assistant  🚩 Fake   
7830                                   Position Finance Assistant  🚩 Fake   
0                                                Marketing Intern  ✅ Real   
20                                            Marketing Assistant  ✅ Real   
2368               Director of Product Marketing for Advertisers   🚩 Fake   

In [10]:
import pandas as pd

# 1. Set display to show the full text without cutting it off
pd.set_option('display.max_colwidth', None)

# 2. Define our target categories
categories = {
    "Data Analyst": "Data Analyst",
    "Software Developer": "Software|Developer|Engineer",
    "Finance": "Finance|Accountant|Banking",
    "Marketing": "Marketing|Manager"
}

results = []

for cat_name, keyword in categories.items():
    # Filter for titles matching the category
    cat_df = df[df['title'].str.contains(keyword, case=False, na=False)]
    
    # Try to get 1 or 2 Fake jobs (fraudulent == 1)
    fakes = cat_df[cat_df['fraudulent'] == 1].head(2)
    
    # Fill the rest with Real jobs (fraudulent == 0) to reach at least 3
    needed_real = 3 - len(fakes)
    reals = cat_df[cat_df['fraudulent'] == 0].head(needed_real)
    
    # Combine and add to our list
    results.append(pd.concat([fakes, reals]))

# 3. Create the final combined DataFrame
mixed_df = pd.concat(results).drop_duplicates()

# 4. Create a readable label for Real vs Fake
mixed_df['Job_Status'] = mixed_df['fraudulent'].map({0: "✅ REAL", 1: "🚩 FAKE"})

# 5. Print the results clearly
for i, row in mixed_df.iterrows():
    print(f"CATEGORY: {row['title']}")
    print(f"STATUS:   {row['Job_Status']}")
    print("-" * 30)
    print(f"FULL DESCRIPTION:\n{row['text_combined']}")
    print("\n" + "="*100 + "\n")

# Summary Table
print("Summary of selected jobs:")
print(mixed_df[['title', 'Job_Status']])

CATEGORY: Data Analyst
STATUS:   ✅ REAL
------------------------------
FULL DESCRIPTION:
Data Analyst Growing out of Forward Internet Group, Scramble has 6 years’ experience of running Internet marketing campaigns. Our focus is on developing smart, automated solutions to maintain our track record of aggressive growth in an ever-more competitive advertising space. We are looking for a Data Analyst to join our growing team to develop and manage PPC accounts.Your day-to-day responsibilities will include using data management tools like Excel and MySQL to review and analyse Adwords data in order to improve the performance of our campaigns.  You will collaborate closely with the rest of the team to help to create new processes and identify opportunities to grow our business. Essential skills and Experience●      Excellent Excel skills; including pivot tables, graphing, vlookups, logic statements and formatting.●      Adwords account management; knowledge of PPC best practice and account per

In [12]:
import pandas as pd

# 1. Set display to show the full text without cutting it off
pd.set_option('display.max_colwidth', None)

# --- NEW STEP: Calculate the length of the text ---
# This creates a new column representing the character count of the combined text
df['text_length'] = df['text_combined'].str.len()

# 2. Define our target categories
categories = {
    "Data Analyst": "Data Analyst",
    "Software Developer": "Software|Developer|Engineer",
    "Finance": "Finance|Accountant|Banking",
    "Marketing": "Marketing|Manager"
}

results = []

for cat_name, keyword in categories.items():
    # Filter for titles matching the category
    cat_df = df[df['title'].str.contains(keyword, case=False, na=False)]
    
    # --- NEW STEP: Sort by length (Ascending = Smallest first) ---
    cat_df = cat_df.sort_values(by='text_length', ascending=True)
    
    # Try to get 1 or 2 Fake jobs (fraudulent == 1)
    # Because we sorted above, .head(2) will now grab the *shortest* fake jobs
    fakes = cat_df[cat_df['fraudulent'] == 1].head(2)
    
    # Fill the rest with Real jobs (fraudulent == 0) to reach at least 3
    # Similarly, this grabs the *shortest* real jobs
    needed_real = 3 - len(fakes)
    reals = cat_df[cat_df['fraudulent'] == 0].head(needed_real)
    
    # Combine and add to our list
    results.append(pd.concat([fakes, reals]))

# 3. Create the final combined DataFrame
mixed_df = pd.concat(results).drop_duplicates()

# 4. Create a readable label for Real vs Fake
mixed_df['Job_Status'] = mixed_df['fraudulent'].map({0: "✅ REAL", 1: "🚩 FAKE"})

# 5. Print the results clearly
print(f"{'CATEGORY':<30} | {'STATUS':<10} | {'LENGTH':<10}")
print("-" * 60)

for i, row in mixed_df.iterrows():
    print(f"{str(row['title'])[:28]:<30} | {row['Job_Status']:<10} | {row['text_length']}")
    print("-" * 30)
    print(f"FULL DESCRIPTION:\n{row['text_combined']}")
    print("\n" + "="*100 + "\n")

# Summary Table
print("Summary of selected jobs (sorted by length):")
print(mixed_df[['title', 'Job_Status', 'text_length']])

CATEGORY                       | STATUS     | LENGTH    
------------------------------------------------------------
Data Analyst                   | ✅ REAL     | 148
------------------------------
FULL DESCRIPTION:
Data Analyst  To work on analytics modelling using desktop and programming tools (R, SPSS) on business data in a financial investment institution.  


Growth Hacker / Data Analyst   | ✅ REAL     | 874
------------------------------
FULL DESCRIPTION:
Growth Hacker / Data Analyst Manager Playfair Capital is an early stage technology investment fund based in London.  Festicket is looking to offer the role of Growth Hacker / Data Analyst Manager to a passionate, analytically minded online marketer who’s a logical but lateral thinker, obsessed by live music and travel, and gets excited by eCommerce and dealing with the world’s coolest festivals and exciting brands. Reporting to Festicket’s Head of Marketing, the successful candidate will be tasked with driving continuous and  i